# Data augmentation
Part 1 applies Keras' random image layers to one photo. Part 2 trains the same small CNN on 2,000 cats-vs-dogs
photos without and with augmentation (3 seeds each) and compares training and validation accuracy.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

## Part 1: one photo, many versions

In [ ]:
import urllib.request
PHOTOS = Path.home() / "datasets" / "photos"

def fetch(url):
    """Download a photo once (Wikimedia asks for a User-Agent) and return its local path."""
    PHOTOS.mkdir(parents=True, exist_ok=True)
    path = PHOTOS / url.split("/")[-1]
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "campusx-notes/1.0 (study notes)"})
        path.write_bytes(urllib.request.urlopen(request).read())
    return path

In [ ]:
# A kitten photo from Wikimedia Commons (David Corby, CC BY 2.5), not part of the dataset.
path = fetch("https://upload.wikimedia.org/wikipedia/commons/b/bb/Kittyply_edit1.jpg")
img = keras.utils.load_img(path, target_size=(200, 200))   # a PIL image, resized to 200 x 200
x = keras.utils.img_to_array(img)                           # a NumPy array (200, 200, 3)
batch = np.expand_dims(x, 0)                                # a batch of one image: (1, 200, 200, 3)
print(type(img).__name__, x.shape, batch.shape)

In [ ]:
# One layer per transformation, with the settings of the Keras blog example (rotation up to 40 degrees,
# shifts, zoom and shear of up to 20%, horizontal flips). RandomRotation takes a fraction of a full turn.
transforms = {
    "flip": layers.RandomFlip("horizontal", seed=1),
    "rotation": layers.RandomRotation(40 / 360, fill_mode="nearest", seed=2),
    "shift": layers.RandomTranslation(0.2, 0.2, fill_mode="nearest", seed=3),
    "zoom": layers.RandomZoom(0.2, fill_mode="nearest", seed=4),
    "shear": layers.RandomShear(0.2, 0.2, fill_mode="nearest", seed=5),
}
combined = keras.Sequential(list(transforms.values()))
OUT = HERE / "data" / "aug"
OUT.mkdir(exist_ok=True)
save = lambda arr, name: keras.utils.save_img(OUT / f"{name}.jpg", np.clip(arr, 0, 255).astype("uint8"),
                                              scale=False, quality=85)
save(x, "original")
for name, layer in transforms.items():
    for k in range(2):
        out = layer(batch, training=True)[0].numpy()
        while name == "flip" and np.allclose(out, x):    # a flip happens only half the time; show one that did
            out = layer(batch, training=True)[0].numpy()
        save(out, f"{name}_{k}")
for k in range(12):                                   # 12 random versions from all five together
    save(combined(batch, training=True)[0].numpy(), f"combined_{k:02d}")
print(sorted(p.name for p in OUT.iterdir()))

In [ ]:
# Fill modes: shift the photo 30% right and down, then fill the empty pixels four ways.
for mode in ("nearest", "reflect", "constant", "wrap"):
    shift = layers.RandomTranslation((0.3, 0.3), (0.3, 0.3), fill_mode=mode)   # a fixed shift, to compare
    save(shift(batch, training=True)[0].numpy(), f"fill_{mode}")
print("saved", len(list(OUT.iterdir())), "small JPEGs,", sum(p.stat().st_size for p in OUT.iterdir()) // 1024, "KB")

In [ ]:
# Outside training the layers do nothing: the photo passes through unchanged.
print("training=False leaves the image unchanged:", bool(np.allclose(combined(batch, training=False), batch)))
a, b = combined(batch, training=True), combined(batch, training=True)
print("two calls with training=True give different images:", not bool(np.allclose(a, b)))

## Part 2: 2,000 training photos, with and without augmentation

In [ ]:
# The Kaggle / Microsoft cats-vs-dogs photos: 12,500 cats and 12,500 dogs, one folder per class.
DATA = Path.home() / "datasets" / "PetImages"
URL = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
if not DATA.exists():
    zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", URL, cache_dir=Path.home(), cache_subdir="datasets")
    import shutil
    shutil.unpack_archive(zip_path, DATA.parent)
# Some files are damaged or are not JPEGs at all; delete them, as the official Keras example does.
removed = 0
for path in DATA.glob("*/*"):
    with open(path, "rb") as f:
        is_jpeg = b"JFIF" in f.peek(10)
    if not is_jpeg:
        path.unlink()
        removed += 1
counts = {d.name: len(list(d.iterdir())) for d in sorted(DATA.iterdir())}
print("removed now:", removed, "| images per class:", counts, "| total:", sum(counts.values()))

In [ ]:
# A small subset, as when data is scarce: 1,000 cats and 1,000 dogs for training,
# 500 + 500 for validation and 500 + 500 for testing, picked at random once (seed 0).
# Symbolic links put them in the folder layout that image_dataset_from_directory expects.
SMALL = Path.home() / "datasets" / "cats_vs_dogs_small"
if not SMALL.exists():
    rng = np.random.default_rng(0)
    for cls in ("Cat", "Dog"):
        files = sorted(DATA.glob(f"{cls}/*.jpg"), key=lambda p: int(p.stem))
        pick = rng.choice(len(files), 2000, replace=False)
        for split, idx in (("train", pick[:1000]), ("validation", pick[1000:1500]), ("test", pick[1500:])):
            folder = SMALL / split / cls.lower()
            folder.mkdir(parents=True, exist_ok=True)
            for i in idx:
                (folder / files[i].name).symlink_to(files[i])
print({split: sum(1 for _ in (SMALL / split).glob("*/*")) for split in ("train", "validation", "test")})

In [ ]:
def load(split, shuffle):
    ds = keras.utils.image_dataset_from_directory(SMALL / split, label_mode="binary", image_size=(150, 150),
                                                  batch_size=16, shuffle=shuffle, seed=1)
    return ds.cache().prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = load("train", True), load("validation", False), load("test", False)

In [ ]:
# The small CNN of the Keras blog post "Building powerful image classification models using very little data".
# Augmentation layers as in Chollet (2021, ch. 8): random horizontal flips, rotations up to 10% of a turn
# (36 degrees) and zooms up to 20%. They sit inside the model, so they act only during training.
def make_model(augment):
    model = keras.Sequential([keras.Input(shape=(150, 150, 3))])
    if augment:
        model.add(layers.RandomFlip("horizontal"))
        model.add(layers.RandomRotation(0.1))
        model.add(layers.RandomZoom(0.2))
    model.add(layers.Rescaling(1 / 255))
    for filters in (32, 32, 64):
        model.add(layers.Conv2D(filters, 3, activation="relu"))
        model.add(layers.MaxPooling2D(2))
    model.add(layers.Flatten())
    model.add(layers.Dense(64, activation="relu"))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])
    return model

make_model(True).summary()

In [ ]:
import time
EPOCHS = 60
rows, test_acc = [], []
for name, flag in (("without", False), ("with", True)):
    for seed in (0, 1, 2):
        keras.utils.set_random_seed(seed)
        model = make_model(flag)
        t = time.time()
        h = model.fit(train_ds, epochs=EPOCHS, validation_data=val_ds, verbose=0)
        for e in range(EPOCHS):
            rows.append(dict(augmentation=name, seed=seed, epoch=e + 1,
                             **{k: h.history[k][e] for k in ("accuracy", "val_accuracy", "loss", "val_loss")}))
        acc = model.evaluate(test_ds, verbose=0)[1]
        test_acc.append(dict(augmentation=name, seed=seed, test_accuracy=acc))
        print(f"{name:7s} seed {seed}: train {h.history['accuracy'][-1]:.3f}  val {h.history['val_accuracy'][-1]:.3f}"
              f"  test {acc:.3f}  ({time.time() - t:.0f} s)")
hist = pd.DataFrame(rows).round(4)
hist.to_csv(HERE / "data" / "history.csv", index=False)
pd.DataFrame(test_acc).round(4).to_csv(HERE / "data" / "test_accuracy.csv", index=False)

In [ ]:
mean = hist.groupby(["augmentation", "epoch"])[["accuracy", "val_accuracy", "loss", "val_loss"]].mean()
mean["gap"] = mean.accuracy - mean.val_accuracy
for name in ("without", "with"):
    g = mean.loc[name]
    print(f"{name} augmentation (mean of 3 seeds)")
    print(g.loc[[5, 10, 25, 40, 60]].round(3).to_string())
    print(f"  best val acc {g.val_accuracy.max():.3f} at epoch {g.val_accuracy.idxmax()};"
          f" lowest val loss {g.val_loss.min():.3f} at epoch {g.val_loss.idxmin()}")
print(pd.DataFrame(test_acc).groupby("augmentation").test_accuracy.agg(["mean", "min", "max"]).round(3))